# Queue delay estimates and confidence intervals

This analysis uses the assumptions in `docs/assumptions.md`: five scenarios, 10 initial replications, two-sided 95% Student-t intervals, and a target half-width equal to 10% of each queue's initial estimated mean delay. A replication's delay is the average for patients who started service in that queue, consistent with the documented terminating-run convention.

For each queue, the required replication count is the smallest `n >= 10` satisfying `t(n - 1, 0.975) * s0 / sqrt(n) <= 0.10 * mean0`, with the pilot mean and standard deviation held fixed. Each scenario uses the larger of its two queue-specific counts as `n*`. The final intervals are recomputed from seeds `1..n*`, reusing the initial replications.

The table puts the initial and `n*` estimates and intervals next to each other. The result table is also saved as `data/queue_delay_confidence_intervals.csv`.


In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))
from queue_delay_analysis import analyze_queue_delays


In [2]:
results = analyze_queue_delays()

rows = []
for row in results.to_dict("records"):
    for queue, label in [("triage", "Triage"), ("room", "Treatment room")]:
        rows.append({
            "Scenario": row["scenario"],
            "Queue": label,
            "Initial mean (min)": f"{row[f'{queue}_initial_mean']:.2f}",
            "Initial 95% CI (min)": (
                f"[{row[f'{queue}_initial_ci_low']:.2f}, "
                f"{row[f'{queue}_initial_ci_high']:.2f}]"
            ),
            "Queue n required": int(row[f"{queue}_required_n"]),
            "Scenario n*": int(row["scenario_n_star"]),
            "n* mean (min)": f"{row[f'{queue}_n_star_mean']:.2f}",
            "n* 95% CI (min)": (
                f"[{row[f'{queue}_n_star_ci_low']:.2f}, "
                f"{row[f'{queue}_n_star_ci_high']:.2f}]"
            ),
        })

comparison = pd.DataFrame(rows)
comparison


Scenario,Queue,Initial mean (min),Initial 95% CI (min),Queue n required,Scenario n*,n* mean (min),n* 95% CI (min)
Base,Triage,180.83,"[156.31, 205.36]",17,162,181.36,"[176.28, 186.44]"
Base,Treatment room,21.78,"[11.75, 31.82]",162,162,19.30,"[17.23, 21.38]"
A,Triage,3.66,"[3.14, 4.19]",19,114,3.83,"[3.62, 4.03]"
A,Treatment room,46.53,"[28.65, 64.40]",114,114,41.74,"[36.44, 47.04]"
B,Triage,3.66,"[3.14, 4.19]",19,534,4.04,"[3.93, 4.15]"
B,Treatment room,0.05,"[0.01, 0.10]",534,534,0.04,"[0.03, 0.04]"
C,Triage,180.83,"[156.31, 205.36]",17,17,172.55,"[156.03, 189.06]"
C,Treatment room,235.02,"[217.97, 252.07]",10,17,231.93,"[220.31, 243.55]"
D,Triage,180.83,"[156.31, 205.36]",17,121,180.74,"[174.78, 186.71]"
D,Treatment room,0.29,"[0.18, 0.41]",121,121,0.25,"[0.22, 0.29]"
